# 🩺 AI Skin Disease Classification & Medical RAG System
## Fine-Tuning EfficientNetB0 CNN on HAM10000 Skin Lesion Dataset

This notebook demonstrates the end-to-end computer vision pipeline for classifying dermatological skin conditions:
1. **Dataset Loading & Preprocessing** (HAM10000 7-class dermatoscopic benchmark)
2. **Data Augmentation & Class Balancing**
3. **Transfer Learning Model Architecture** (EfficientNetB0)
4. **Training & Fine-Tuning**
5. **Evaluation Metrics** (Confusion Matrix, Precision, Recall, AUC-ROC)
6. **Exporting Model Artifacts** (`skin_disease_classifier.keras`)

In [ ]:
import os
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import tensorflow as tf
from tensorflow.keras import layers, models
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from sklearn.metrics import classification_report, confusion_matrix

print(f"TensorFlow Version: {tf.__version__}")
print(f"GPU Available: {tf.config.list_physical_devices('GPU')}")

### Step 1: Dataset Directory Setup & Class Mapping

In [ ]:
DATA_DIR = '../dataset/HAM10000'
CLASS_MAP_PATH = '../class_names.json'

with open(CLASS_MAP_PATH, 'r') as f:
    class_info = json.load(f)

print("Skin Disease Classes:")
for code, info in class_info.items():
    print(f"  - {code.upper()}: {info['name']} (Severity: {info['severity']})")

### Step 2: Data Generators & Augmentation

In [ ]:
IMG_SIZE = (224, 224)
BATCH_SIZE = 32

datagen = ImageDataGenerator(
    preprocessing_function=tf.keras.applications.efficientnet.preprocess_input,
    rotation_range=30,
    width_shift_range=0.15,
    height_shift_range=0.15,
    shear_range=0.15,
    zoom_range=0.2,
    horizontal_flip=True,
    vertical_flip=True,
    validation_split=0.2
)

if os.path.exists(DATA_DIR):
    train_gen = datagen.flow_from_directory(
        DATA_DIR, target_size=IMG_SIZE, batch_size=BATCH_SIZE,
        class_mode='categorical', subset='training'
    )
    val_gen = datagen.flow_from_directory(
        DATA_DIR, target_size=IMG_SIZE, batch_size=BATCH_SIZE,
        class_mode='categorical', subset='validation'
    )
else:
    print("Dataset folder empty. Please download HAM10000 using `download_dataset.py`.")

### Step 3: Build EfficientNetB0 Transfer Learning Architecture

In [ ]:
def create_efficientnet_model(num_classes=7):
    base_model = tf.keras.applications.EfficientNetB0(
        include_top=False, weights='imagenet', input_shape=(224, 224, 3)
    )
    base_model.trainable = False

    inputs = tf.keras.Input(shape=(224, 224, 3))
    x = base_model(inputs, training=False)
    x = layers.GlobalAveragePooling2D()(x)
    x = layers.BatchNormalization()(x)
    x = layers.Dropout(0.4)(x)
    x = layers.Dense(256, activation='relu')(x)
    x = layers.Dropout(0.3)(x)
    outputs = layers.Dense(num_classes, activation='softmax')(x)

    model = models.Model(inputs, outputs)
    model.compile(
        optimizer=tf.keras.optimizers.Adam(1e-3),
        loss='categorical_crossentropy',
        metrics=['accuracy', tf.keras.metrics.AUC(name='auc')]
    )
    return model

model = create_efficientnet_model()
model.summary()

### Step 4: Model Training & Fine-Tuning

In [ ]:
# Callbacks
callbacks = [
    tf.keras.callbacks.EarlyStopping(monitor='val_loss', patience=4, restore_best_weights=True),
    tf.keras.callbacks.ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=2)
]

# Execute training if dataset exists
if 'train_gen' in locals():
    history = model.fit(
        train_gen, epochs=10, validation_data=val_gen, callbacks=callbacks
    )
    # Save Model
    model.save('../models/skin_disease_classifier.keras')
    print("Model saved to models/skin_disease_classifier.keras")